In [1]:
# ============================
# ETL COMPLETO - LIQUOR BRANDS
# ============================

import pandas as pd
import re
import html
from google.colab import files

# ============================
# 1. EXTRACT - SUBIR Y CARGAR CSV
# ============================

df = pd.read_csv("https://raw.githubusercontent.com/AlejandraG7/Proyecto-BI-AlejandraGonzalez/refs/heads/main/Data/CSV/Liquor_Brands.csv")

df.head()
print("Archivo cargado correctamente")
print("Dimensiones iniciales:", df.shape)

# Revisar cantidad de filas y columnas
print("Dimensiones del dataset:")
print(df.shape)

# Revisar nombres de columnas y tipos de datos
print("\nInformación general:")
print(df.info())

# Revisar valores nulos
print("\nValores nulos:")
print(df.isnull().sum())

# Revisar duplicados
print("\nRegistros duplicados:")
print(df.duplicated().sum())

# ============================
# 2. TRANSFORM - LIMPIEZA
# ============================

# Renombramos columnas para que sean más fáciles de trabajar en Python
df = df.rename(columns={
    "BRAND-NAME": "brand_name",
    "CT-REGISTRATION-NUMBER": "ct_registration_number",
    "STATUS": "status",
    "EFFECTIVE": "effective_date",
    "EXPIRATION": "expiration_date",
    "OUT-OF-STATE-SHIPPER": "out_of_state_shipper",
    "SUPERVISOR-CREDENTIAL-": "supervisor_credential",
    "WHOLESALERS": "wholesalers"
})

# Verificamos los nuevos nombres de columnas
df.columns

# Eliminamos registros duplicados exactos
df = df.drop_duplicates().copy()

# Columnas de texto que se van a limpiar
text_columns = [
    "brand_name",
    "ct_registration_number",
    "status",
    "out_of_state_shipper",
    "supervisor_credential",
    "wholesalers"
]

# Limpiar espacios, saltos innecesarios y caracteres especiales HTML
for col in text_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Convertir símbolos HTML como &amp; en texto normal
    df[col] = df[col].apply(lambda x: html.unescape(x) if pd.notna(x) else x)

# Revisamos las primeras filas después de limpiar texto
df.head()

# Convertimos las columnas de fechas a formato datetime
df["effective_date"] = pd.to_datetime(
    df["effective_date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

df["expiration_date"] = pd.to_datetime(
    df["expiration_date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

# Revisamos los tipos de datos después de convertir fechas
df.info()

# Rellenamos valores faltantes en columnas de texto
df["out_of_state_shipper"] = df["out_of_state_shipper"].fillna("NO REGISTRADO")
df["supervisor_credential"] = df["supervisor_credential"].fillna("NO REGISTRADO")
df["wholesalers"] = df["wholesalers"].fillna("NO REGISTRADO")

# Verificamos nuevamente los valores nulos
df.isnull().sum()

# Fecha actual para calcular vencimientos
today = pd.Timestamp.today().normalize()

# Crear año de inicio de vigencia
df["effective_year"] = df["effective_date"].dt.year

# Crear año de expiración
df["expiration_year"] = df["expiration_date"].dt.year

# Calcular duración de la licencia en días
df["license_duration_days"] = (
    df["expiration_date"] - df["effective_date"]
).dt.days

# Calcular cuántos días faltan para vencer
df["days_to_expiration"] = (
    df["expiration_date"] - today
).dt.days

# Identificar si el registro ya está vencido
df["is_expired"] = df["expiration_date"].lt(today)

# Identificar si tiene mayorista registrado
df["has_wholesaler"] = df["wholesalers"].ne("NO REGISTRADO")

# Mostrar primeras filas con las nuevas columnas
df.head()

# Función para contar cuántos mayoristas tiene cada registro
def count_wholesalers(value):
    if pd.isna(value) or value == "NO REGISTRADO":
        return 0

    # Busca códigos con formato LIW. seguido de números
    return len(re.findall(r"\((LIW\.\d+)\)", value))

# Crear columna con cantidad de mayoristas
df["wholesaler_count"] = df["wholesalers"].apply(count_wholesalers)

# Revisar resultado
df[["brand_name", "wholesalers", "wholesaler_count"]].head()

# ============================
# NORMALIZAR MAYORISTAS
# ============================

# Lista donde se guardarán los mayoristas separados
wholesaler_rows = []

# Recorremos cada fila del dataset
for _, row in df.iterrows():
    wholesaler_text = row["wholesalers"]

    # Si no tiene mayorista, se omite
    if pd.isna(wholesaler_text) or wholesaler_text == "NO REGISTRADO":
        continue

    # Extraer nombre del mayorista y su credencial
    matches = re.finditer(r"([^()]+?)\s*\((LIW\.\d+)\)", wholesaler_text)

    for match in matches:
        wholesaler_name = match.group(1).strip(" ,")
        wholesaler_credential = match.group(2).strip()

        wholesaler_rows.append({
            "ct_registration_number": row["ct_registration_number"],
            "brand_name": row["brand_name"],
            "wholesaler_name": wholesaler_name,
            "wholesaler_credential": wholesaler_credential
        })

# Crear DataFrame normalizado
df_wholesalers = pd.DataFrame(wholesaler_rows)

# Eliminar duplicados en la tabla de mayoristas
df_wholesalers = df_wholesalers.drop_duplicates().reset_index(drop=True)

# Mostrar primeras filas
df_wholesalers.head()

# Ordenamos las columnas finales del archivo limpio
columns_order = [
    "ct_registration_number",
    "brand_name",
    "status",
    "effective_date",
    "expiration_date",
    "effective_year",
    "expiration_year",
    "license_duration_days",
    "days_to_expiration",
    "is_expired",
    "out_of_state_shipper",
    "supervisor_credential",
    "wholesalers",
    "has_wholesaler",
    "wholesaler_count"
]

# Crear DataFrame final
df_final = df[columns_order].copy()

# Revisar primeras filas del dataset final
df_final.head()

# ============================
# 3. LOAD - GUARDAR ARCHIVOS
# ============================

# Guardar el archivo principal limpio
df_final.to_csv("liquor_brands_etl_limpio.csv", index=False, encoding="utf-8-sig")

# Guardar la tabla normalizada de mayoristas
df_wholesalers.to_csv("liquor_brand_wholesalers_normalizado.csv", index=False, encoding="utf-8-sig")

print("ETL ejecutado correctamente.")
print("Archivo principal generado: liquor_brands_etl_limpio.csv")
print("Archivo de mayoristas generado: liquor_brand_wholesalers_normalizado.csv")
print("Registros finales:", len(df_final))
print("Registros de mayoristas:", len(df_wholesalers))

Archivo cargado correctamente
Dimensiones iniciales: (71578, 8)
Dimensiones del dataset:
(71578, 8)

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71578 entries, 0 to 71577
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   BRAND-NAME              71578 non-null  object
 1   CT-REGISTRATION-NUMBER  71578 non-null  object
 2   STATUS                  71578 non-null  object
 3   EFFECTIVE               71567 non-null  object
 4   EXPIRATION              71567 non-null  object
 5   OUT-OF-STATE-SHIPPER    71517 non-null  object
 6   SUPERVISOR-CREDENTIAL-  71517 non-null  object
 7   WHOLESALERS             60478 non-null  object
dtypes: object(8)
memory usage: 4.4+ MB
None

Valores nulos:
BRAND-NAME                    0
CT-REGISTRATION-NUMBER        0
STATUS                        0
EFFECTIVE                    11
EXPIRATION                   11
OUT-OF-STATE-SHIPPER         

## 4. Modelo estrella: dimensiones, hechos y tabla puente

Este bloque genera los CSV necesarios para cargar el modelo analítico en SQL Server o Power BI.


In [2]:
# ==========================================================
# 4. CREACIÓN DE DIMENSIONES, TABLA DE HECHOS Y TABLA PUENTE
# ==========================================================
# Este bloque se ejecuta después del ETL principal, cuando ya existen:
# - df_final: dataset limpio principal
# - df_wholesalers: tabla normalizada de mayoristas

import pandas as pd
from IPython.display import display

# ----------------------------------------------------------
# 4.1 Preparar copia de trabajo
# ----------------------------------------------------------

df_modelo = df_final.copy()

# Convertir fechas por seguridad
df_modelo["effective_date"] = pd.to_datetime(df_modelo["effective_date"], errors="coerce")
df_modelo["expiration_date"] = pd.to_datetime(df_modelo["expiration_date"], errors="coerce")

# Limpiar campos de texto usados como llaves
text_key_columns = [
    "ct_registration_number",
    "brand_name",
    "status",
    "out_of_state_shipper",
    "supervisor_credential"
]

for col in text_key_columns:
    df_modelo[col] = (
        df_modelo[col]
        .fillna("NO REGISTRADO")
        .astype(str)
        .str.strip()
    )

# Ordenar para crear un ID estable del registro
df_modelo = (
    df_modelo
    .sort_values(["ct_registration_number", "brand_name", "effective_date"])
    .reset_index(drop=True)
)

# ID central del registro para la tabla de hechos y la tabla puente
df_modelo["id_registro"] = range(1, len(df_modelo) + 1)

# ----------------------------------------------------------
# 4.2 Clasificación de estado de control
# ----------------------------------------------------------

def es_texto_invalido(valor):
    """Valida textos vacíos o no registrados."""
    if pd.isna(valor):
        return True

    valor = str(valor).strip().upper()
    return valor in ["", "NO REGISTRADO", "NAN", "NONE", "<NA>"]

def clasificar_control(row):
    """
    Clasifica cada registro para análisis de fiscalización.
    Puedes cambiar 60 por 30 si deseas considerar 'próximo a vencer'
    solo con un mes de anticipación.
    """
    estado = str(row["status"]).strip().upper()
    dias = row["days_to_expiration"]

    if (
        es_texto_invalido(row["brand_name"])
        or es_texto_invalido(row["ct_registration_number"])
        or pd.isna(row["effective_date"])
        or pd.isna(row["expiration_date"])
    ):
        return "INCOMPLETO"

    if "EXPIRED" in estado or "CANCEL" in estado or (pd.notna(dias) and dias < 0):
        return "VENCIDO"

    if pd.notna(dias) and 0 <= dias <= 60:
        return "PROXIMO A VENCER"

    if "ACTIVE" in estado or (pd.notna(dias) and dias > 60):
        return "ACTIVO"

    return "REVISAR"

df_modelo["categoria_control"] = df_modelo.apply(clasificar_control, axis=1)

# ----------------------------------------------------------
# 4.3 DimTiempo
# ----------------------------------------------------------

# Se crea una dimensión de tiempo con dos tipos de fecha:
# INICIO_VIGENCIA y VENCIMIENTO.
fechas_inicio = (
    df_modelo[["effective_date"]]
    .dropna()
    .rename(columns={"effective_date": "fecha"})
)
fechas_inicio["tipo_fecha"] = "INICIO_VIGENCIA"

fechas_vencimiento = (
    df_modelo[["expiration_date"]]
    .dropna()
    .rename(columns={"expiration_date": "fecha"})
)
fechas_vencimiento["tipo_fecha"] = "VENCIMIENTO"

dim_tiempo = pd.concat(
    [fechas_inicio, fechas_vencimiento],
    ignore_index=True
)

dim_tiempo["fecha"] = pd.to_datetime(dim_tiempo["fecha"], errors="coerce").dt.normalize()

dim_tiempo = (
    dim_tiempo
    .drop_duplicates(["fecha", "tipo_fecha"])
    .sort_values(["fecha", "tipo_fecha"])
    .reset_index(drop=True)
)

dim_tiempo.insert(0, "id_tiempo", range(1, len(dim_tiempo) + 1))

meses_es = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

dim_tiempo["dia"] = dim_tiempo["fecha"].dt.day
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses_es)
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter
dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year

dim_tiempo = dim_tiempo[
    [
        "id_tiempo",
        "fecha",
        "dia",
        "mes",
        "nombre_mes",
        "trimestre",
        "anio",
        "tipo_fecha"
    ]
]

# ----------------------------------------------------------
# 4.4 DimMarcaLicor
# ----------------------------------------------------------

dim_marca = (
    df_modelo[["brand_name", "ct_registration_number", "status"]]
    .drop_duplicates()
    .sort_values(["brand_name", "ct_registration_number"])
    .reset_index(drop=True)
)

dim_marca.insert(0, "id_marca", range(1, len(dim_marca) + 1))

dim_marca = dim_marca.rename(columns={
    "brand_name": "nombre_marca",
    "ct_registration_number": "numero_registro",
    "status": "estado_registro"
})

dim_marca = dim_marca[
    [
        "id_marca",
        "nombre_marca",
        "numero_registro",
        "estado_registro"
    ]
]

# ----------------------------------------------------------
# 4.5 DimProveedor
# ----------------------------------------------------------

dim_proveedor = (
    df_modelo[["out_of_state_shipper", "supervisor_credential"]]
    .drop_duplicates()
    .sort_values(["out_of_state_shipper", "supervisor_credential"])
    .reset_index(drop=True)
)

dim_proveedor.insert(0, "id_proveedor", range(1, len(dim_proveedor) + 1))

dim_proveedor = dim_proveedor.rename(columns={
    "out_of_state_shipper": "proveedor_remitente",
    "supervisor_credential": "credencial_supervisor"
})

dim_proveedor = dim_proveedor[
    [
        "id_proveedor",
        "proveedor_remitente",
        "credencial_supervisor"
    ]
]

# ----------------------------------------------------------
# 4.6 DimMayorista
# ----------------------------------------------------------

# Validar que exista df_wholesalers.
# Si no existe o está vacío, se crea una estructura vacía para evitar errores.
if "df_wholesalers" in globals():
    df_mayoristas_base = df_wholesalers.copy()
else:
    df_mayoristas_base = pd.DataFrame(columns=[
        "ct_registration_number",
        "brand_name",
        "wholesaler_name",
        "wholesaler_credential"
    ])

required_wholesaler_columns = [
    "ct_registration_number",
    "brand_name",
    "wholesaler_name",
    "wholesaler_credential"
]

for col in required_wholesaler_columns:
    if col not in df_mayoristas_base.columns:
        df_mayoristas_base[col] = pd.Series(dtype="string")

for col in required_wholesaler_columns:
    df_mayoristas_base[col] = (
        df_mayoristas_base[col]
        .fillna("NO REGISTRADO")
        .astype(str)
        .str.strip()
    )

dim_mayorista = (
    df_mayoristas_base[["wholesaler_name", "wholesaler_credential"]]
    .drop_duplicates()
    .sort_values(["wholesaler_name", "wholesaler_credential"])
    .reset_index(drop=True)
)

dim_mayorista.insert(0, "id_mayorista", range(1, len(dim_mayorista) + 1))

dim_mayorista = dim_mayorista.rename(columns={
    "wholesaler_name": "nombre_mayorista",
    "wholesaler_credential": "licencia_mayorista"
})

dim_mayorista = dim_mayorista[
    [
        "id_mayorista",
        "nombre_mayorista",
        "licencia_mayorista"
    ]
]

# ----------------------------------------------------------
# 4.7 DimEstadoRegistro
# ----------------------------------------------------------

dim_estado = (
    df_modelo[["status", "categoria_control"]]
    .drop_duplicates()
    .sort_values(["status", "categoria_control"])
    .reset_index(drop=True)
)

dim_estado.insert(0, "id_estado", range(1, len(dim_estado) + 1))

dim_estado = dim_estado.rename(columns={
    "status": "estado_registro"
})

dim_estado = dim_estado[
    [
        "id_estado",
        "estado_registro",
        "categoria_control"
    ]
]

# ----------------------------------------------------------
# 4.8 Relacionar llaves foráneas para FactRegistroLicor
# ----------------------------------------------------------

# Llave de tiempo inicio
tiempo_inicio = (
    dim_tiempo[dim_tiempo["tipo_fecha"] == "INICIO_VIGENCIA"][["id_tiempo", "fecha"]]
    .rename(columns={
        "id_tiempo": "id_tiempo_inicio",
        "fecha": "fecha_inicio_key"
    })
)

df_modelo["fecha_inicio_key"] = df_modelo["effective_date"].dt.normalize()

df_modelo = df_modelo.merge(
    tiempo_inicio,
    on="fecha_inicio_key",
    how="left"
)

# Llave de tiempo vencimiento
tiempo_vencimiento = (
    dim_tiempo[dim_tiempo["tipo_fecha"] == "VENCIMIENTO"][["id_tiempo", "fecha"]]
    .rename(columns={
        "id_tiempo": "id_tiempo_vencimiento",
        "fecha": "fecha_vencimiento_key"
    })
)

df_modelo["fecha_vencimiento_key"] = df_modelo["expiration_date"].dt.normalize()

df_modelo = df_modelo.merge(
    tiempo_vencimiento,
    on="fecha_vencimiento_key",
    how="left"
)

# Llave de marca
marca_key = dim_marca.rename(columns={
    "nombre_marca": "brand_name",
    "numero_registro": "ct_registration_number",
    "estado_registro": "status"
})

df_modelo = df_modelo.merge(
    marca_key[["id_marca", "brand_name", "ct_registration_number", "status"]],
    on=["brand_name", "ct_registration_number", "status"],
    how="left"
)

# Llave de proveedor
proveedor_key = dim_proveedor.rename(columns={
    "proveedor_remitente": "out_of_state_shipper",
    "credencial_supervisor": "supervisor_credential"
})

df_modelo = df_modelo.merge(
    proveedor_key[["id_proveedor", "out_of_state_shipper", "supervisor_credential"]],
    on=["out_of_state_shipper", "supervisor_credential"],
    how="left"
)

# Llave de estado
estado_key = dim_estado.rename(columns={
    "estado_registro": "status"
})

df_modelo = df_modelo.merge(
    estado_key[["id_estado", "status", "categoria_control"]],
    on=["status", "categoria_control"],
    how="left"
)

# ----------------------------------------------------------
# 4.9 FactRegistroLicor
# ----------------------------------------------------------

# Asegurar métricas numéricas
df_modelo["license_duration_days"] = pd.to_numeric(
    df_modelo["license_duration_days"],
    errors="coerce"
).astype("Int64")

df_modelo["days_to_expiration"] = pd.to_numeric(
    df_modelo["days_to_expiration"],
    errors="coerce"
).astype("Int64")

df_modelo["wholesaler_count"] = pd.to_numeric(
    df_modelo["wholesaler_count"],
    errors="coerce"
).fillna(0).astype(int)

# Campo binario para SQL Server y Power BI
df_modelo["tiene_mayorista"] = (df_modelo["wholesaler_count"] > 0).astype(int)

fact_registro_licor = df_modelo[
    [
        "id_registro",
        "id_tiempo_inicio",
        "id_tiempo_vencimiento",
        "id_marca",
        "id_proveedor",
        "id_estado",
        "license_duration_days",
        "days_to_expiration",
        "tiene_mayorista",
        "wholesaler_count"
    ]
].copy()

fact_registro_licor = fact_registro_licor.rename(columns={
    "license_duration_days": "dias_vigencia",
    "days_to_expiration": "dias_para_vencer",
    "wholesaler_count": "cantidad_mayoristas"
})

# Convertir llaves a enteros compatibles con nulos
key_columns = [
    "id_tiempo_inicio",
    "id_tiempo_vencimiento",
    "id_marca",
    "id_proveedor",
    "id_estado"
]

for col in key_columns:
    fact_registro_licor[col] = fact_registro_licor[col].astype("Int64")

# ----------------------------------------------------------
# 4.10 BridgeRegistroMayorista
# ----------------------------------------------------------

if len(df_mayoristas_base) > 0 and len(dim_mayorista) > 0:
    registros_key = df_modelo[
        [
            "id_registro",
            "ct_registration_number",
            "brand_name"
        ]
    ].drop_duplicates()

    mayorista_key = dim_mayorista.rename(columns={
        "nombre_mayorista": "wholesaler_name",
        "licencia_mayorista": "wholesaler_credential"
    })

    bridge_registro_mayorista = df_mayoristas_base.merge(
        registros_key,
        on=["ct_registration_number", "brand_name"],
        how="left"
    )

    bridge_registro_mayorista = bridge_registro_mayorista.merge(
        mayorista_key[["id_mayorista", "wholesaler_name", "wholesaler_credential"]],
        on=["wholesaler_name", "wholesaler_credential"],
        how="left"
    )

    bridge_registro_mayorista = (
        bridge_registro_mayorista[["id_registro", "id_mayorista"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["id_registro", "id_mayorista"])
        .reset_index(drop=True)
    )

    bridge_registro_mayorista["id_registro"] = bridge_registro_mayorista["id_registro"].astype("Int64")
    bridge_registro_mayorista["id_mayorista"] = bridge_registro_mayorista["id_mayorista"].astype("Int64")

else:
    bridge_registro_mayorista = pd.DataFrame(columns=[
        "id_registro",
        "id_mayorista"
    ])

# ----------------------------------------------------------
# 4.11 Exportar CSV para cargar en SQL Server
# ----------------------------------------------------------

# Formatear fecha para que SQL Server la lea mejor
dim_tiempo_export = dim_tiempo.copy()
dim_tiempo_export["fecha"] = dim_tiempo_export["fecha"].dt.strftime("%Y-%m-%d")

dim_tiempo_export.to_csv("DimTiempo.csv", index=False, encoding="utf-8-sig")
dim_marca.to_csv("DimMarcaLicor.csv", index=False, encoding="utf-8-sig")
dim_proveedor.to_csv("DimProveedor.csv", index=False, encoding="utf-8-sig")
dim_mayorista.to_csv("DimMayorista.csv", index=False, encoding="utf-8-sig")
dim_estado.to_csv("DimEstadoRegistro.csv", index=False, encoding="utf-8-sig")
fact_registro_licor.to_csv("FactRegistroLicor.csv", index=False, encoding="utf-8-sig")
bridge_registro_mayorista.to_csv("BridgeRegistroMayorista.csv", index=False, encoding="utf-8-sig")

print("Modelo estrella generado correctamente.")
print("DimTiempo:", len(dim_tiempo_export))
print("DimMarcaLicor:", len(dim_marca))
print("DimProveedor:", len(dim_proveedor))
print("DimMayorista:", len(dim_mayorista))
print("DimEstadoRegistro:", len(dim_estado))
print("FactRegistroLicor:", len(fact_registro_licor))
print("BridgeRegistroMayorista:", len(bridge_registro_mayorista))

# Mostrar vistas rápidas
display(dim_tiempo_export.head())
display(dim_marca.head())
display(dim_proveedor.head())
display(dim_mayorista.head())
display(dim_estado.head())
display(fact_registro_licor.head())
display(bridge_registro_mayorista.head())


Modelo estrella generado correctamente.
DimTiempo: 2368
DimMarcaLicor: 71578
DimProveedor: 1836
DimMayorista: 110
DimEstadoRegistro: 4
FactRegistroLicor: 71578
BridgeRegistroMayorista: 79295


,id_tiempo,fecha,dia,mes,nombre_mes,trimestre,anio,tipo_fecha
0,1,2014-04-03,3,4,Abril,2,2014,INICIO_VIGENCIA
1,2,2020-12-15,15,12,Diciembre,4,2020,INICIO_VIGENCIA
2,3,2020-12-16,16,12,Diciembre,4,2020,INICIO_VIGENCIA
3,4,2021-07-20,20,7,Julio,3,2021,INICIO_VIGENCIA
4,5,2021-09-02,2,9,Septiembre,3,2021,INICIO_VIGENCIA


,id_marca,nombre_marca,numero_registro,estado_registro
0,1,#ADULTING,LBD.0128591,ACTIVE
1,2,#GVLTAT,LBD.0204784,ACTIVE
2,3,#LOVEISLOVE,LBD.0211940,ACTIVE
3,4,'23 BHEEYO WINE,LBD.0211603,ACTIVE
4,5,'23 MONJE WINE,LBD.0211604,ACTIVE


,id_proveedor,proveedor_remitente,credencial_supervisor
0,1,'MERICAN MULE,LSL.0001760
1,2,001 VINTNERS LLC,LSW.0001793
2,3,1-800 WINESHOP.COM INC,LSW.0000432
3,4,1185 LLC,LSL.0002016
4,5,1260 SUMMIT LAKE LLC,LSW.0001742


,id_mayorista,nombre_mayorista,licencia_mayorista
0,1,,LIW.0000543
1,2,,LIW.0000639
2,3,504 IMPORT & DISTRIBUTION LLC,LIW.0000658
3,4,55 DEGREES DISTRIBUTION LLC,LIW.0000649
4,5,A GALLO COMPANY OF LITCHFIELD,LIW.0000540


,id_estado,estado_registro,categoria_control
0,1,ACTIVE,ACTIVO
1,2,ACTIVE,INCOMPLETO
2,3,ACTIVE,PROXIMO A VENCER
3,4,ACTIVE,VENCIDO


,id_registro,id_tiempo_inicio,id_tiempo_vencimiento,id_marca,id_proveedor,id_estado,dias_vigencia,dias_para_vencer,tiene_mayorista,cantidad_mayoristas
0,1,95,1253,71572,78,3,1094,57,0,0
1,2,679,1837,40611,79,1,1094,641,1,1
2,3,95,1253,52860,79,3,1094,57,0,0
3,4,140,1299,40448,1076,1,1094,102,0,0
4,5,140,1299,14200,1076,1,1094,102,0,0


,id_registro,id_mayorista
0,2,43
1,6,23
2,6,44
3,6,97
4,7,23
